# Support-Sumset Geometry

Before carry, the raw support of multiplication is the support sumset

$$S_{raw} = S_X + S_Y.$$

The diagonal density

$$\rho_k = \#\{(i,j): i+j=k\}$$

counts how many digit-support pairs can contribute to degree `k`. This controls the potential raw mass `c_k`, while digit values determine the actual mass.

In [ ]:
from pathlib import Path
import sys

sys.path.append("../src")
if (Path.cwd() / "src").exists():
    sys.path.append(str(Path.cwd() / "src"))
elif (Path.cwd().parent / "src").exists():
    sys.path.append(str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd
from experiments import experiment_support_geometry_carry
from support_geometry import additive_energy, representation_counts, support_sumset, sumset_summary

## Dense and sparse support demos

In [ ]:
dense = set(range(6))
sparse = {0, 5, 10}

for name, S in [("dense", dense), ("sparse", sparse)]:
    counts = representation_counts(S, S)
    print(name)
    print("support:", sorted(S))
    print("sumset:", sorted(support_sumset(S, S)))
    print("rho_k:", counts)
    print("additive_energy:", additive_energy(S, S))
    print("summary:", sumset_summary(S, S))
    print()

In [ ]:
RESULTS = Path("results") if (Path.cwd() / "results").exists() else Path("../results")
RESULTS.mkdir(parents=True, exist_ok=True)
csv_path = RESULTS / "support_geometry_carry.csv"

if csv_path.exists():
    data = pd.read_csv(csv_path)
else:
    data = experiment_support_geometry_carry(
        n_trials=20,
        num_digits=10,
        B=10,
        densities=[0.2, 0.5, 0.7, 1.0],
        output_csv=str(csv_path),
    )

data.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for density, group in data.groupby("density"):
    axes[0, 0].scatter(
        group["max_representation_count"],
        group["carry_mass"],
        alpha=0.65,
        label=f"density={density}",
    )
axes[0, 0].set_xlabel("max_representation_count")
axes[0, 0].set_ylabel("carry_mass")
axes[0, 0].set_title("max rho_k vs carry mass")
axes[0, 0].legend()

axes[0, 1].scatter(data["additive_energy"], data["carry_mass"], alpha=0.65, color="#E45756")
axes[0, 1].set_xlabel("additive_energy")
axes[0, 1].set_ylabel("carry_mass")
axes[0, 1].set_title("additive energy vs carry mass")

axes[1, 0].scatter(data["size_sumset"], data["carry_count"], alpha=0.65, color="#54A24B")
axes[1, 0].set_xlabel("size_sumset")
axes[1, 0].set_ylabel("carry_count")
axes[1, 0].set_title("sumset size vs carry count")

axes[1, 1].scatter(data["support_density_x"], data["carry_amplification_ratio"], alpha=0.55, label="Sx")
axes[1, 1].scatter(data["support_density_y"], data["carry_amplification_ratio"], alpha=0.55, label="Sy")
axes[1, 1].set_xlabel("support density")
axes[1, 1].set_ylabel("carry_amplification_ratio")
axes[1, 1].set_title("support density vs amplification")
axes[1, 1].legend()

for ax in axes.ravel():
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

Additive energy measures concentration of representation counts. High energy means many support-pair collisions on shared anti-diagonals, which can raise potential raw mass and make carry flow more likely. Digit values still matter, so support geometry is a predictor rather than the whole story.